In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/sweettv-tv-program-recommender/export_arh_31-42-final.csv
/kaggle/input/competitions/sweettv-tv-program-recommender/export_arh_21-30-final.csv
/kaggle/input/competitions/sweettv-tv-program-recommender/export_arh_11-20-final.csv
/kaggle/input/competitions/sweettv-tv-program-recommender/submission.csv
/kaggle/input/competitions/sweettv-tv-program-recommender/dataset11-30.csv


In [2]:
import csv
from pathlib import Path
from itertools import islice

BASE_PATH = Path(
    "/kaggle/input/competitions/sweettv-tv-program-recommender"
)

FILES = {
    "dataset11-30.csv": {"user_id", "vsetv_id", "start_time"},
    "export_arh_11-20-final.csv": {
        "channel_id", "start_time", "duration", "tv_show_id"
    },
    "export_arh_21-30-final.csv": {
        "channel_id", "start_time", "duration", "tv_show_id"
    },
    "export_arh_31-42-final.csv": {
        "channel_id", "start_time", "duration", "tv_show_id"
    },
    "submission.csv": {"user_id", "tv_show_id"},
}

DELIMITERS = [",", ";", "\t", "|"]

print("SWEET.TV — HEADERS AND FIRST THREE RECORDS")
print("Values are displayed as strings. No files are modified.")

for filename, expected_columns in FILES.items():
    path = BASE_PATH / filename

    print("\n" + "=" * 80)
    print(f"FILE: {filename}")

    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    matches = []

    for delimiter in DELIMITERS:
        with path.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle, delimiter=delimiter)
            header = next(reader, [])

        normalized_header = {
            column.strip().lower() for column in header
        }

        if expected_columns.issubset(normalized_header):
            matches.append((delimiter, header))

    if len(matches) != 1:
        raise ValueError(
            f"{filename}: expected one valid delimiter, "
            f"found {len(matches)}. Candidates: {matches}"
        )

    delimiter, header = matches[0]

    print(f"Delimiter: {delimiter!r}")
    print(f"Columns ({len(header)}): {header}")

    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, delimiter=delimiter)
        next(reader)

        for row_number, values in enumerate(islice(reader, 3), start=1):
            if len(values) != len(header):
                raise ValueError(
                    f"{filename}, record {row_number}: "
                    f"{len(values)} values for {len(header)} columns"
                )

            print(f"\nRecord {row_number}:")
            for column, value in zip(header, values):
                print(f"  {column}: {value!r}")

print("\n" + "=" * 80)
print("STRUCTURE CHECK COMPLETE")

SWEET.TV — HEADERS AND FIRST THREE RECORDS
Values are displayed as strings. No files are modified.

FILE: dataset11-30.csv
Delimiter: ','
Columns (5): ['user_id', 'vsetv_id', 'start_time', 'stop_time', 'duraton']

Record 1:
  user_id: '3398461054087191302'
  vsetv_id: '6'
  start_time: '2020-03-09 00:00:02'
  stop_time: '2020-03-09 00:01:53'
  duraton: '111'

Record 2:
  user_id: '17226860011138219284'
  vsetv_id: '353'
  start_time: '2020-03-09 00:00:02'
  stop_time: '2020-03-09 00:03:26'
  duraton: '204'

Record 3:
  user_id: '18417531283109304442'
  vsetv_id: '332'
  start_time: '2020-03-09 00:00:05'
  stop_time: '2020-03-09 00:01:26'
  duraton: '81'

FILE: export_arh_11-20-final.csv
Delimiter: ','
Columns (13): ['channel_id', 'channel_title', 'start_time', 'duration', 'tv_show_title', 'tv_show_id', 'tv_show_category', 'tv_show_genre_1', 'tv_show_genre_2', 'tv_show_genre_3', 'year_of_production', 'director', 'actors']

Record 1:
  channel_id: '3'
  channel_title: '1+1'
  start_time:

In [8]:
import json
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path(
    "/kaggle/input/competitions/sweettv-tv-program-recommender"
)
OUTPUT = Path("/kaggle/working/data_diagnostic_summary.json")
CHUNK_SIZE = 100_000

FILES = {
    "viewing_log": "dataset11-30.csv",
    "schedule_11_20": "export_arh_11-20-final.csv",
    "schedule_21_30": "export_arh_21-30-final.csv",
    "schedule_31_42": "export_arh_31-42-final.csv",
    "submission": "submission.csv",
}

VIEW_FORMAT = "%Y-%m-%d %H:%M:%S"
SCHEDULE_FORMAT = "%d.%m.%Y %H:%M:%S"


def clean_strings(series):
    return series.str.strip()


def missing_mask(series):
    return series.isna() | series.eq("")


def update_time_stats(stats, name, raw, date_format):
    raw = clean_strings(raw)
    missing = missing_mask(raw)

    parsed = pd.to_datetime(
        raw,
        format=date_format,
        errors="coerce",
        exact=True,
    )

    item = stats.setdefault(
        name,
        {
            "missing": 0,
            "parse_errors": 0,
            "min": None,
            "max": None,
            "error_examples": [],
        },
    )

    invalid = (~missing) & parsed.isna()
    item["missing"] += int(missing.sum())
    item["parse_errors"] += int(invalid.sum())

    remaining = 5 - len(item["error_examples"])
    if remaining > 0:
        item["error_examples"].extend(
            raw[invalid].head(remaining).tolist()
        )

    if parsed.notna().any():
        chunk_min = parsed.min()
        chunk_max = parsed.max()

        if item["min"] is None or chunk_min < item["min"]:
            item["min"] = chunk_min

        if item["max"] is None or chunk_max > item["max"]:
            item["max"] = chunk_max

    return parsed


def analyze_file(key, path):
    is_viewing = key == "viewing_log"

    required = (
        {"user_id", "vsetv_id", "start_time", "stop_time", "duraton"}
        if is_viewing
        else {"channel_id", "start_time", "duration", "tv_show_id"}
    )

    header = pd.read_csv(
        path,
        nrows=0,
        encoding="utf-8-sig",
    ).columns.tolist()

    absent = required - set(header)
    if absent:
        raise ValueError(
            f"{path.name}: missing required columns {sorted(absent)}"
        )

    id_columns = (
        ["user_id", "vsetv_id"]
        if is_viewing
        else ["channel_id", "tv_show_id"]
    )

    unique_ids = {column: set() for column in id_columns}
    missing_ids = {column: 0 for column in id_columns}
    time_stats = {}
    duration_arrays = []

    result = {
        "filename": path.name,
        "columns": header,
        "records": 0,
        "missing_ids": missing_ids,
        "duration_missing": 0,
        "duration_invalid_or_nonfinite": 0,
        "duration_negative": 0,
        "duration_zero": 0,
    }

    if is_viewing:
        result.update(
            {
                "duration_checked": 0,
                "duration_matches": 0,
                "duration_mismatches": 0,
                "stop_before_start": 0,
            }
        )
    else:
        result["tv_show_id_zero"] = 0

    duration_column = "duraton" if is_viewing else "duration"
    date_format = VIEW_FORMAT if is_viewing else SCHEDULE_FORMAT

    reader = pd.read_csv(
        path,
        usecols=sorted(required),
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        chunksize=CHUNK_SIZE,
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        result["records"] += len(chunk)

        for column in id_columns:
            values = clean_strings(chunk[column])
            missing = missing_mask(values)
            missing_ids[column] += int(missing.sum())
            unique_ids[column].update(values[~missing].unique())

        start = update_time_stats(
            time_stats,
            "start_time",
            chunk["start_time"],
            date_format,
        )

        duration_raw = clean_strings(chunk[duration_column])
        duration_missing = missing_mask(duration_raw)
        duration = pd.to_numeric(duration_raw, errors="coerce")
        finite = np.isfinite(duration)

        result["duration_missing"] += int(duration_missing.sum())
        result["duration_invalid_or_nonfinite"] += int(
            ((~duration_missing) & (~finite)).sum()
        )
        result["duration_negative"] += int(
            (finite & duration.lt(0)).sum()
        )
        result["duration_zero"] += int(
            (finite & duration.eq(0)).sum()
        )

        duration_arrays.append(
            duration[finite].to_numpy(dtype=np.float64, copy=True)
        )

        if is_viewing:
            stop = update_time_stats(
                time_stats,
                "stop_time",
                chunk["stop_time"],
                date_format,
            )

            valid_times = start.notna() & stop.notna()
            result["stop_before_start"] += int(
                (valid_times & stop.lt(start)).sum()
            )

            checkable = valid_times & finite
            difference = (stop - start).dt.total_seconds()
            matches = checkable & difference.eq(duration)

            result["duration_checked"] += int(checkable.sum())
            result["duration_matches"] += int(matches.sum())
            result["duration_mismatches"] += int(
                (checkable & (~matches)).sum()
            )
        else:
            programme_ids = clean_strings(chunk["tv_show_id"])
            result["tv_show_id_zero"] += int(
                programme_ids.eq("0").sum()
            )

        if chunk_number % 10 == 0:
            print(
                f"  {path.name}: "
                f"{result['records']:,} records processed",
                flush=True,
            )

    result["unique_ids"] = {
        column: len(values)
        for column, values in unique_ids.items()
    }

    if not is_viewing:
        result["unique_nonzero_programme_ids"] = len(
            unique_ids["tv_show_id"] - {"0"}
        )
        result["tv_show_id_zero_fraction"] = (
            result["tv_show_id_zero"] / result["records"]
            if result["records"]
            else None
        )

    for item in time_stats.values():
        for boundary in ["min", "max"]:
            if item[boundary] is not None:
                item[boundary] = item[boundary].isoformat()

    result["time"] = time_stats

    durations = (
        np.concatenate(duration_arrays)
        if duration_arrays
        else np.array([], dtype=np.float64)
    )

    result["duration_stats_full_file"] = {
        "finite_count": int(durations.size),
        "min": float(durations.min()) if durations.size else None,
        "max": float(durations.max()) if durations.size else None,
        "median": float(np.median(durations)) if durations.size else None,
        "mean": float(durations.mean()) if durations.size else None,
    }

    return result


def analyze_submission(path):
    frame = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    if list(frame.columns) != ["user_id", "tv_show_id"]:
        raise ValueError(
            f"Unexpected submission columns: {list(frame.columns)}"
        )

    users = clean_strings(frame["user_id"])

    return {
        "filename": path.name,
        "columns": list(frame.columns),
        "records": int(len(frame)),
        "missing_user_ids": int(missing_mask(users).sum()),
        "duplicate_user_ids": int(users.duplicated().sum()),
        "user_ids_with_surrounding_whitespace": int(
            frame["user_id"].ne(users).sum()
        ),
        "sample_rows": frame.head(3).to_dict("records"),
    }


def main():
    started = time.perf_counter()

    summary = {
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "timezone": "Not specified in source files; no conversion applied",
        "files": {},
        "warnings": [],
    }

    print("SWEET.TV — FULL DIAGNOSTIC", flush=True)
    print(
        f"Python {platform.python_version()} | "
        f"pandas {pd.__version__} | NumPy {np.__version__}",
        flush=True,
    )

    for key, filename in FILES.items():
        path = BASE / filename
        if not path.is_file():
            raise FileNotFoundError(str(path))

        print(f"\nProcessing: {filename}", flush=True)
        file_started = time.perf_counter()

        result = (
            analyze_submission(path)
            if key == "submission"
            else analyze_file(key, path)
        )

        result["runtime_seconds"] = round(
            time.perf_counter() - file_started, 3
        )
        summary["files"][key] = result

        print(
            json.dumps(
                result,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            ),
            flush=True,
        )

        if key != "submission":
            if any(result["missing_ids"].values()):
                summary["warnings"].append(
                    f"{key}: missing identifiers detected"
                )

            for column, info in result["time"].items():
                if info["missing"] or info["parse_errors"]:
                    summary["warnings"].append(
                        f"{key}: missing or invalid {column}"
                    )

            if any(
                result[field] > 0
                for field in [
                    "duration_missing",
                    "duration_invalid_or_nonfinite",
                    "duration_negative",
                    "duration_zero",
                ]
            ):
                summary["warnings"].append(
                    f"{key}: missing, invalid or nonpositive durations"
                )

            if key == "viewing_log":
                if result["duration_mismatches"]:
                    summary["warnings"].append(
                        "viewing_log: duration mismatches detected"
                    )
                if result["stop_before_start"]:
                    summary["warnings"].append(
                        "viewing_log: stop precedes start"
                    )
        else:
            if result["missing_user_ids"] or result["duplicate_user_ids"]:
                summary["warnings"].append(
                    "submission: missing or duplicate user IDs"
                )

    summary["runtime_seconds"] = round(
        time.perf_counter() - started, 3
    )

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT.open("w", encoding="utf-8") as handle:
        json.dump(
            summary,
            handle,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    print("\n" + "=" * 70)
    print("DIAGNOSTIC COMPLETE")
    print(f"Total runtime: {summary['runtime_seconds']} seconds")
    print(f"Saved: {OUTPUT}")
    print("Warnings:", summary["warnings"])
    print("Interval join and model training have NOT been run.")

    return summary


diagnostic_summary = main()

SWEET.TV — FULL DIAGNOSTIC
Python 3.13.15 | pandas 2.3.3 | NumPy 2.1.3

Processing: dataset11-30.csv
  dataset11-30.csv: 1,000,000 records processed
  dataset11-30.csv: 2,000,000 records processed
  dataset11-30.csv: 3,000,000 records processed
  dataset11-30.csv: 4,000,000 records processed
  dataset11-30.csv: 5,000,000 records processed
{
  "filename": "dataset11-30.csv",
  "columns": [
    "user_id",
    "vsetv_id",
    "start_time",
    "stop_time",
    "duraton"
  ],
  "records": 5821987,
  "missing_ids": {
    "user_id": 0,
    "vsetv_id": 0
  },
  "duration_missing": 0,
  "duration_invalid_or_nonfinite": 0,
  "duration_negative": 0,
  "duration_zero": 0,
  "duration_checked": 5821987,
  "duration_matches": 5821870,
  "duration_mismatches": 117,
  "stop_before_start": 0,
  "unique_ids": {
    "user_id": 4914,
    "vsetv_id": 202
  },
  "time": {
    "start_time": {
      "missing": 0,
      "parse_errors": 0,
      "min": "2020-03-09T00:00:02",
      "max": "2020-07-26T23:58:42",

In [9]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path(
    "/kaggle/input/competitions/sweettv-tv-program-recommender"
)
OUT = Path("/kaggle/working")
CHUNK_SIZE = 100_000

SCHEDULE_FILES = [
    "export_arh_11-20-final.csv",
    "export_arh_21-30-final.csv",
    "export_arh_31-42-final.csv",
]

ZONES = {
    "may": {
        "start": pd.Timestamp("2020-05-17"),
        "end": pd.Timestamp("2020-05-19"),
        "split": pd.Timestamp("2020-05-18"),
        "files": SCHEDULE_FILES[:2],
    },
    "july": {
        "start": pd.Timestamp("2020-07-26"),
        "end": pd.Timestamp("2020-07-28"),
        "split": pd.Timestamp("2020-07-27"),
        "files": SCHEDULE_FILES[1:],
    },
}

started = time.perf_counter()
summary = {
    "timezone": "Unspecified; no timezone conversion applied",
    "duplicate_definition": (
        "Same channel_id, parsed start_time, numeric duration, "
        "and tv_show_id; metadata equality is not implied"
    ),
}

print("SWEET.TV — ANOMALY AND BOUNDARY CHECKS", flush=True)

# 1. Inspect viewing-duration mismatches.
mismatch_parts = []
checked = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        BASE / "dataset11-30.csv",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        chunksize=CHUNK_SIZE,
    ),
    start=1,
):
    start = pd.to_datetime(
        chunk["start_time"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )
    stop = pd.to_datetime(
        chunk["stop_time"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )
    recorded = pd.to_numeric(chunk["duraton"], errors="raise")
    computed = (stop - start).dt.total_seconds()
    difference = recorded - computed
    mask = difference.ne(0)

    checked += len(chunk)

    if mask.any():
        selected = chunk.loc[mask].copy()
        selected["computed_seconds"] = computed.loc[mask]
        selected["recorded_seconds"] = recorded.loc[mask]
        selected["recorded_minus_computed"] = difference.loc[mask]
        mismatch_parts.append(selected)

    if chunk_number % 10 == 0:
        print(f"Viewing records checked: {checked:,}", flush=True)

mismatches = (
    pd.concat(mismatch_parts, ignore_index=True)
    if mismatch_parts
    else pd.DataFrame(
        columns=[
            "user_id", "vsetv_id", "start_time", "stop_time",
            "duraton", "computed_seconds", "recorded_seconds",
            "recorded_minus_computed",
        ]
    )
)

mismatches.to_csv(OUT / "duration_mismatches.csv", index=False)

difference_counts = (
    mismatches["recorded_minus_computed"]
    .value_counts()
    .sort_index()
)

summary["duration_mismatches"] = {
    "records_checked": checked,
    "mismatch_count": len(mismatches),
    "difference_definition": "recorded duraton minus interval seconds",
    "difference_distribution": [
        {"difference_seconds": float(value), "records": int(count)}
        for value, count in difference_counts.items()
    ],
    "min_difference": (
        float(mismatches["recorded_minus_computed"].min())
        if len(mismatches) else None
    ),
    "max_difference": (
        float(mismatches["recorded_minus_computed"].max())
        if len(mismatches) else None
    ),
    "max_absolute_difference": (
        float(mismatches["recorded_minus_computed"].abs().max())
        if len(mismatches) else None
    ),
}

print("\nDURATION MISMATCH SUMMARY")
print(json.dumps(summary["duration_mismatches"], indent=2))
print("\nFIRST 10 MISMATCHES")
print(mismatches.head(10).to_string(index=False))

# 2. Collect zero-duration rows and boundary-zone records.
zero_parts = []
zone_parts = {name: [] for name in ZONES}

for filename in SCHEDULE_FILES:
    print(f"\nScanning schedule: {filename}", flush=True)

    for chunk in pd.read_csv(
        BASE / filename,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        chunksize=CHUNK_SIZE,
    ):
        start = pd.to_datetime(
            chunk["start_time"],
            format="%d.%m.%Y %H:%M:%S",
            errors="raise",
        )
        duration = pd.to_numeric(chunk["duration"], errors="raise")

        if not np.isfinite(duration).all():
            raise ValueError(f"Nonfinite schedule duration: {filename}")

        end = start + pd.to_timedelta(duration, unit="s")

        zero_mask = duration.eq(0)
        if zero_mask.any():
            selected = chunk.loc[zero_mask].copy()
            selected["_source"] = filename
            zero_parts.append(selected)

        for name, zone in ZONES.items():
            if filename not in zone["files"]:
                continue

            mask = (
                duration.gt(0)
                & start.lt(zone["end"])
                & end.gt(zone["start"])
            )

            if mask.any():
                selected = chunk.loc[mask].copy()
                selected["_start"] = start.loc[mask]
                selected["_end"] = end.loc[mask]
                selected["_duration"] = duration.loc[mask]
                selected["_source"] = filename
                zone_parts[name].append(selected)

zero_rows = (
    pd.concat(zero_parts, ignore_index=True)
    if zero_parts else pd.DataFrame()
)
summary["zero_duration_rows"] = zero_rows.to_dict("records")

print("\nZERO-DURATION SCHEDULE ROWS")
print(
    zero_rows.to_string(index=False)
    if len(zero_rows) else "None"
)

# 3. Compare airing identities, conflicts, and split crossings.
summary["boundaries"] = {}

for name, zone in ZONES.items():
    if not zone_parts[name]:
        summary["boundaries"][name] = {"records": 0}
        continue

    frame = pd.concat(zone_parts[name], ignore_index=True)

    identity = [
        "channel_id", "_start", "_duration", "tv_show_id"
    ]

    # Repeated airing identities across source files.
    identity_groups = (
        frame.groupby(identity, dropna=False)
        .agg(
            records=("_source", "size"),
            source_count=("_source", "nunique"),
        )
        .reset_index()
    )

    cross_file_duplicates = identity_groups[
        identity_groups["source_count"].gt(1)
    ]

    # Different durations or IDs at the same channel/start.
    variants = frame.drop_duplicates(identity)
    variant_counts = (
        variants.groupby(["channel_id", "_start"], dropna=False)
        .size()
        .reset_index(name="variant_count")
    )
    conflicting_keys = variant_counts[
        variant_counts["variant_count"].gt(1)
    ]

    conflicts = frame.merge(
        conflicting_keys,
        on=["channel_id", "_start"],
        how="inner",
    )

    # Count both raw rows and unique airing identities.
    crossings = frame[
        frame["_start"].lt(zone["split"])
        & frame["_end"].gt(zone["split"])
    ]
    unique_crossings = crossings.drop_duplicates(identity)

    frame.to_csv(OUT / f"{name}_boundary_records.csv", index=False)
    conflicts.to_csv(
        OUT / f"{name}_boundary_conflicts.csv", index=False
    )
    crossings.to_csv(
        OUT / f"{name}_boundary_crossings.csv", index=False
    )
    cross_file_duplicates.to_csv(
        OUT / f"{name}_cross_file_duplicate_airings.csv",
        index=False,
    )

    result = {
        "zone_start": zone["start"].isoformat(),
        "zone_end": zone["end"].isoformat(),
        "split": zone["split"].isoformat(),
        "source_record_counts": {
            str(source): int(count)
            for source, count in frame["_source"].value_counts().items()
        },
        "cross_file_duplicate_airing_groups": len(
            cross_file_duplicates
        ),
        "conflicting_channel_start_groups": len(conflicting_keys),
        "crossing_rows_before_deduplication": len(crossings),
        "unique_crossing_airing_identities": len(unique_crossings),
    }

    summary["boundaries"][name] = result

    print(f"\n{name.upper()} BOUNDARY SUMMARY")
    print(json.dumps(result, indent=2))

    print("\nDuplicate airing examples:")
    print(cross_file_duplicates.head(5).to_string(index=False))

    print("\nConflict examples:")
    print(
        conflicts[
            ["channel_id", "start_time", "duration",
             "tv_show_id", "_source"]
        ].head(10).to_string(index=False)
    )

    print("\nCrossing examples:")
    print(
        unique_crossings[
            ["channel_id", "_start", "_end",
             "tv_show_id", "_source"]
        ].head(10).to_string(index=False)
    )

summary["runtime_seconds"] = round(
    time.perf_counter() - started, 3
)

with (OUT / "boundary_and_anomaly_checks.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(
        summary,
        handle,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("\nINVESTIGATION COMPLETE")
print(f"Runtime: {summary['runtime_seconds']} seconds")
print("Results saved in /kaggle/working")
print("No source rows were modified or removed.")
print("No viewing-to-schedule join or model training was performed.")

SWEET.TV — ANOMALY AND BOUNDARY CHECKS
Viewing records checked: 1,000,000
Viewing records checked: 2,000,000
Viewing records checked: 3,000,000
Viewing records checked: 4,000,000
Viewing records checked: 5,000,000

DURATION MISMATCH SUMMARY
{
  "records_checked": 5821987,
  "mismatch_count": 117,
  "difference_definition": "recorded duraton minus interval seconds",
  "difference_distribution": [
    {
      "difference_seconds": -3600.0,
      "records": 117
    }
  ],
  "min_difference": -3600.0,
  "max_difference": -3600.0,
  "max_absolute_difference": 3600.0
}

FIRST 10 MISMATCHES
             user_id vsetv_id          start_time           stop_time duraton  computed_seconds  recorded_seconds  recorded_minus_computed
12466582247545246052     1521 2020-03-28 22:47:45 2020-03-30 09:02:48  119703          123303.0            119703                  -3600.0
10059425067810734795        7 2020-03-28 22:56:59 2020-03-29 04:11:55   15296           18896.0             15296                  

In [11]:
# Fix categorical grouping and rerun the existing build pipeline.

import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


def join_channel(view_rows, airing_rows):
    """
    Accumulate watched seconds per airing across merged viewing intervals.
    Inputs refer to a single channel.
    """
    if view_rows.empty or airing_rows.empty:
        return Counter()

    airings = airing_rows.sort_values(
        ["start", "stop", "item"],
        kind="stable",
    ).reset_index(drop=True)

    starts = airings["start"].to_numpy(dtype=np.int64)
    stops = airings["stop"].to_numpy(dtype=np.int64)
    durations = stops - starts
    items = airings["item"].to_numpy()

    assert (durations > 0).all()
    max_duration = int(durations.max())
    counts = Counter()

    for user, history in view_rows.groupby(
        "user",
        sort=False,
        observed=True,
    ):
        intervals = merge_intervals(
            zip(history["start"], history["stop"])
        )
        watched = defaultdict(int)

        for view_start, view_stop in intervals:
            left = int(
                np.searchsorted(
                    starts,
                    view_start - max_duration,
                    side="left",
                )
            )
            right = int(
                np.searchsorted(
                    starts,
                    view_stop,
                    side="left",
                )
            )

            for index in range(left, right):
                overlap = (
                    min(view_stop, int(stops[index]))
                    - max(view_start, int(starts[index]))
                )

                if overlap > 0:
                    watched[index] += overlap

        for index, watched_seconds in watched.items():
            duration = int(durations[index])
            assert 0 < watched_seconds <= duration

            if 5 * watched_seconds >= 4 * duration:
                counts[(str(user), str(items[index]))] += 1

    return counts


def build_counts(views, airings, window, progress=False):
    lower, upper = window

    selected_views = views.loc[
        views["start"].lt(upper)
        & views["stop"].gt(lower)
    ].copy()

    selected_views["start"] = selected_views["start"].clip(
        lower=lower
    )
    selected_views["stop"] = selected_views["stop"].clip(
        upper=upper
    )

    selected_airings = airings.loc[
        airings["start"].ge(lower)
        & airings["stop"].le(upper)
        & airings["stop"].gt(airings["start"])
    ]

    if selected_views.empty or selected_airings.empty:
        return Counter()

    view_groups = selected_views.groupby(
        "channel",
        sort=False,
        observed=True,
    )
    airing_groups = selected_airings.groupby(
        "channel",
        sort=False,
        observed=True,
    )

    # Use actual observed values, not retained categorical levels.
    channels = sorted(
        set(selected_views["channel"].unique())
        & set(selected_airings["channel"].unique())
    )

    result = Counter()
    started = time.perf_counter()

    for number, channel in enumerate(channels, start=1):
        result.update(
            join_channel(
                view_groups.get_group(channel),
                airing_groups.get_group(channel),
            )
        )

        if progress and (
            number == 1
            or number % 10 == 0
            or number == len(channels)
        ):
            print(
                f"  Channels {number}/{len(channels)} | "
                f"pairs {len(result):,} | "
                f"{time.perf_counter() - started:.1f}s",
                flush=True,
            )

    return result


# Regression test: an unused category must not produce a missing group.
_regression_views = pd.DataFrame(
    {
        "user": pd.Categorical(
            ["u"],
            categories=["u", "unused_user"],
        ),
        "channel": pd.Categorical(
            ["active"],
            categories=["active", "1172"],
        ),
        "start": [0],
        "stop": [1000],
    }
)

_regression_airings = pd.DataFrame(
    {
        "channel": ["active", "1172"],
        "item": ["1", "2"],
        "start": [0, 0],
        "stop": [1000, 1000],
    }
)

assert dict(
    build_counts(
        _regression_views,
        _regression_airings,
        (0, 2000),
    )
) == {("u", "1"): 1}

print("PASS: unused categorical channel regression", flush=True)

# Reuse the previously defined loading, testing and saving functions.
# Rebuild both splits so the final summary describes one complete run.
interaction_build_summary = main()

PASS: unused categorical channel regression
SWEET.TV — INTERACTION BUILD
PASS: exactly_80_percent
PASS: below_80_percent
PASS: two_fragments_sum_to_80
PASS: no_double_counting
PASS: touching_endpoints
PASS: different_channels
PASS: long_view_multiple_airings
PASS: repeat_airings_count_twice
PASS: overlapping_airings
PASS: window_clipping
PASS: fragments_from_separate_chunks

Synthetic tests passed: 11
Schedule loaded: export_arh_11-20-final.csv
Schedule loaded: export_arh_21-30-final.csv
{
  "sources": [
    {
      "file": "export_arh_11-20-final.csv",
      "records": 1074235,
      "excluded_nonpositive_duration": 1,
      "excluded_zero_id_after_duration_filter": 958621
    },
    {
      "file": "export_arh_21-30-final.csv",
      "records": 1074394,
      "excluded_nonpositive_duration": 0,
      "excluded_zero_id_after_duration_filter": 952889
    }
  ],
  "duplicates_removed": 0,
  "conflict_groups": 0,
  "unique_boundary_crossing_excluded": 131,
  "entirely_outside": 281,
  "t

In [12]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

WORK = Path("/kaggle/working")
INPUT = Path(
    "/kaggle/input/competitions/sweettv-tv-program-recommender"
)

started = time.perf_counter()


def ap_at_5(relevant, predictions):
    relevant = set(relevant)

    if not relevant:
        raise ValueError("AP@5 requires a nonempty relevant set.")

    hits = 0
    total = 0.0
    seen = set()

    for rank, item in enumerate(predictions[:5], start=1):
        if item in seen:
            continue
        seen.add(item)

        if item in relevant:
            hits += 1
            total += hits / rank

    return total / min(5, len(relevant))


# Hand-checkable metric tests.
assert ap_at_5(["a", "b"], ["a", "b", "x", "y", "z"]) == 1.0

assert np.isclose(
    ap_at_5(["a", "b"], ["x", "a", "y", "b", "z"]),
    (1 / 2 + 2 / 4) / 2,
)

assert np.isclose(
    ap_at_5(["a", "b"], ["a", "a", "x", "y", "z"]),
    1 / 2,
)

assert ap_at_5(["a"], ["x", "y", "z", "w", "v"]) == 0.0

print("PASS: four AP@5 checks", flush=True)

train = pd.read_csv(
    WORK / "train_interactions.csv",
    dtype={"user_id": str, "tv_show_id": str},
)
validation = pd.read_csv(
    WORK / "validation_interactions.csv",
    dtype={"user_id": str, "tv_show_id": str},
)
candidates = pd.read_csv(
    WORK / "validation_candidate_ids.csv",
    dtype=str,
)["tv_show_id"].tolist()

template = pd.read_csv(
    INPUT / "submission.csv",
    dtype=str,
    keep_default_na=False,
)

for name, frame in [("train", train), ("validation", validation)]:
    assert not frame[["user_id", "tv_show_id"]].isna().any().any()
    assert not frame.duplicated(["user_id", "tv_show_id"]).any()
    assert frame["watch_count"].gt(0).all()
    assert frame["watch_count"].eq(
        np.floor(frame["watch_count"])
    ).all()
    assert frame["tv_show_id"].ne("0").all()

assert len(candidates) == len(set(candidates))
assert "0" not in candidates
assert len(candidates) >= 5
assert template["user_id"].is_unique

candidate_set = set(candidates)
submission_users = template["user_id"].tolist()
train_users = set(train["user_id"])
validation_users = set(validation["user_id"])

# Fix the cohort before comparing any personalized methods.
# Include cold-start users when they have a nonempty validation target.
cohort = [
    user for user in submission_users
    if user in validation_users
]

if not cohort:
    raise ValueError(
        "No submission users have validation interactions. "
        "Stop and review the evaluation protocol."
    )

# Ground truth: up to five most frequently watched programmes.
# Exact frequency ties are resolved by ascending numeric programme ID.
validation = validation.copy()
validation["_item_numeric"] = validation["tv_show_id"].map(int)

ordered_validation = validation.sort_values(
    ["user_id", "watch_count", "_item_numeric"],
    ascending=[True, False, True],
    kind="stable",
)

targets = (
    ordered_validation.groupby("user_id", sort=False)
    .head(5)
    .groupby("user_id", sort=False)["tv_show_id"]
    .agg(list)
    .to_dict()
)

# Train-only popularity: total qualifying airing views per programme.
popularity = (
    train.groupby("tv_show_id", sort=False)["watch_count"]
    .sum()
    .to_dict()
)

ranked_candidates = sorted(
    candidates,
    key=lambda item: (
        -int(popularity.get(item, 0)),
        int(item),
    ),
)
top5 = ranked_candidates[:5]

assert len(top5) == 5
assert len(set(top5)) == 5
assert set(top5).issubset(candidate_set)

rows = []
recommendation_rows = []

for user in cohort:
    relevant = targets[user]
    unreachable = [
        item for item in relevant
        if item not in candidate_set
    ]

    rows.append(
        {
            "user_id": user,
            "has_train_history": user in train_users,
            "relevant_count": len(relevant),
            "relevant_outside_candidate_pool": len(unreachable),
            "ap_at_5": ap_at_5(relevant, top5),
            "relevant_ids": " ".join(relevant),
            "recommended_ids": " ".join(top5),
        }
    )

    for rank, item in enumerate(top5, start=1):
        recommendation_rows.append(
            {
                "user_id": user,
                "method": "popularity",
                "rank": rank,
                "tv_show_id": item,
                "score": int(popularity.get(item, 0)),
            }
        )

per_user = pd.DataFrame(rows)

warm = per_user.loc[per_user["has_train_history"]]
cold = per_user.loc[~per_user["has_train_history"]]

summary = {
    "method": "popularity",
    "popularity_definition": (
        "Sum of qualifying train-airing views per programme"
    ),
    "metric": "MAP@5",
    "ap_denominator": "min(5, number of unique target programmes)",
    "target_definition": (
        "Top five by validation watch_count; "
        "frequency ties use ascending numeric programme ID"
    ),
    "cohort_definition": (
        "Submission-template users with nonempty validation targets, "
        "including users without train interactions"
    ),
    "repeats_of_previously_watched_items_allowed": True,
    "submission_users": len(submission_users),
    "cohort_size": len(cohort),
    "submission_users_without_validation_target": (
        len(submission_users) - len(cohort)
    ),
    "validation_candidate_programmes": len(candidates),
    "warm_users": len(warm),
    "cold_users": len(cold),
    "map_at_5": float(per_user["ap_at_5"].mean()),
    "warm_map_at_5": (
        float(warm["ap_at_5"].mean()) if len(warm) else None
    ),
    "cold_map_at_5": (
        float(cold["ap_at_5"].mean()) if len(cold) else None
    ),
    "users_with_fewer_than_five_targets": int(
        per_user["relevant_count"].lt(5).sum()
    ),
    "target_occurrences_outside_candidate_pool": int(
        per_user["relevant_outside_candidate_pool"].sum()
    ),
    "top5_ids": top5,
    "top5_train_popularity": [
        int(popularity.get(item, 0)) for item in top5
    ],
    "zero_popularity_fillers_in_top5": sum(
        popularity.get(item, 0) == 0 for item in top5
    ),
    "metric_tests_passed": 4,
    "runtime_seconds": round(time.perf_counter() - started, 3),
}

pd.DataFrame({"user_id": cohort}).to_csv(
    WORK / "validation_cohort.csv",
    index=False,
)

pd.DataFrame(
    [
        {
            "user_id": user,
            "relevant_ids": " ".join(targets[user]),
        }
        for user in cohort
    ]
).to_csv(
    WORK / "validation_targets.csv",
    index=False,
)

per_user.to_csv(
    WORK / "popularity_validation_per_user.csv",
    index=False,
)

pd.DataFrame(recommendation_rows).to_csv(
    WORK / "popularity_validation_recommendations.csv",
    index=False,
)

with (WORK / "popularity_validation_summary.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(
        summary,
        handle,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("\nPOPULARITY VALIDATION COMPLETE")
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\nSaved:")
for filename in [
    "validation_cohort.csv",
    "validation_targets.csv",
    "popularity_validation_per_user.csv",
    "popularity_validation_recommendations.csv",
    "popularity_validation_summary.json",
]:
    print(f"  {filename}")

print("\nNo content-based or PPR model has been fitted.")

PASS: four AP@5 checks

POPULARITY VALIDATION COMPLETE
{
  "method": "popularity",
  "popularity_definition": "Sum of qualifying train-airing views per programme",
  "metric": "MAP@5",
  "ap_denominator": "min(5, number of unique target programmes)",
  "target_definition": "Top five by validation watch_count; frequency ties use ascending numeric programme ID",
  "cohort_definition": "Submission-template users with nonempty validation targets, including users without train interactions",
  "repeats_of_previously_watched_items_allowed": true,
  "submission_users": 1260,
  "cohort_size": 1238,
  "submission_users_without_validation_target": 22,
  "validation_candidate_programmes": 6370,
  "warm_users": 1224,
  "cold_users": 14,
  "map_at_5": 0.2961265930712619,
  "warm_map_at_5": 0.29763457697893975,
  "cold_map_at_5": 0.1642857142857143,
  "users_with_fewer_than_five_targets": 88,
  "target_occurrences_outside_candidate_pool": 0,
  "top5_ids": [
    "2400480",
    "240081",
    "12001682

In [13]:
import json
import re
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.sparse import csr_matrix, vstack
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.preprocessing import normalize

WORK = Path("/kaggle/working")
started = time.perf_counter()

# Fixed initial settings; not claimed to be validation-optimal.
FIELD_WEIGHTS = {
    "category": 2.0,
    "genre": 3.0,
    "year": 0.5,
    "director": 1.0,
    "actor": 0.5,
}


def ap5(relevant, predictions):
    relevant = set(relevant)
    if not relevant:
        raise ValueError("Empty target.")

    hits = 0
    total = 0.0
    seen = set()

    for rank, item in enumerate(predictions[:5], start=1):
        if item in seen:
            continue
        seen.add(item)
        if item in relevant:
            hits += 1
            total += hits / rank

    return total / min(5, len(relevant))


def clean_value(value):
    return " ".join(str(value).strip().casefold().split())


def metadata_features(row):
    features = {}

    def add(field, value):
        value = clean_value(value)
        if value:
            features[f"{field}={value}"] = 1.0

    add("category", row.get("tv_show_category", ""))

    for column in [
        "tv_show_genre_1",
        "tv_show_genre_2",
        "tv_show_genre_3",
    ]:
        add("genre", row.get(column, ""))

    add("year", row.get("year_of_production", ""))

    for field, column in [
        ("director", "director"),
        ("actor", "actors"),
    ]:
        raw = row.get(column, "")
        for name in re.split(r"[,;]", str(raw)):
            add(field, name)

    return features


print("CONTENT-BASED — VALIDATION", flush=True)

train = pd.read_csv(
    WORK / "train_interactions.csv",
    dtype={"user_id": str, "tv_show_id": str},
)
metadata = pd.read_csv(
    WORK / "train_item_metadata.csv",
    dtype=str,
    keep_default_na=False,
)
cohort = pd.read_csv(
    WORK / "validation_cohort.csv", dtype=str
)["user_id"].tolist()

target_frame = pd.read_csv(
    WORK / "validation_targets.csv",
    dtype=str,
    keep_default_na=False,
)
targets = {
    row.user_id: row.relevant_ids.split()
    for row in target_frame.itertuples(index=False)
}

candidate_ids = sorted(
    pd.read_csv(
        WORK / "validation_candidate_ids.csv",
        dtype=str,
    )["tv_show_id"].tolist(),
    key=int,
)

assert len(candidate_ids) == len(set(candidate_ids))
assert len(candidate_ids) >= 5
assert "0" not in candidate_ids
assert metadata["tv_show_id"].is_unique
assert len(cohort) == len(set(cohort))
assert all(targets.get(user) for user in cohort)

popularity = (
    train.groupby("tv_show_id")["watch_count"].sum().to_dict()
)
popularity_order = sorted(
    candidate_ids,
    key=lambda item: (-int(popularity.get(item, 0)), int(item)),
)
popularity_top5 = popularity_order[:5]

metadata = metadata.sort_values(
    "tv_show_id", key=lambda column: column.map(int)
).reset_index(drop=True)

item_ids = metadata["tv_show_id"].tolist()
item_index = {
    item: index for index, item in enumerate(item_ids)
}

feature_dicts = [
    metadata_features(row)
    for row in metadata.to_dict("records")
]

vectorizer = DictVectorizer(sparse=True, dtype=np.float64)
raw_items = vectorizer.fit_transform(feature_dicts)

if raw_items.shape[1] == 0:
    raise ValueError("No usable train metadata features.")

tfidf = TfidfTransformer(norm=None, smooth_idf=True)
item_vectors = tfidf.fit_transform(raw_items)

feature_weights = np.array(
    [
        FIELD_WEIGHTS[name.split("=", 1)[0]]
        for name in vectorizer.get_feature_names_out()
    ],
    dtype=np.float64,
)

item_vectors = normalize(
    item_vectors.multiply(feature_weights).tocsr(),
    norm="l2",
    axis=1,
).tocsr()

# Candidate representations are looked up only from train metadata.
# No validation metadata are read or fitted.
zero_row = csr_matrix(
    (1, item_vectors.shape[1]), dtype=np.float64
)
candidate_vectors = vstack(
    [
        item_vectors.getrow(item_index[item])
        if item in item_index else zero_row
        for item in candidate_ids
    ],
    format="csr",
)

train_users = sorted(train["user_id"].unique(), key=int)
user_index = {
    user: index for index, user in enumerate(train_users)
}

unknown_history_items = (
    set(train["tv_show_id"]) - set(item_index)
)
if unknown_history_items:
    raise ValueError(
        "Train interactions contain items without train metadata."
    )

history = csr_matrix(
    (
        np.log1p(
            train["watch_count"].to_numpy(dtype=np.float64)
        ),
        (
            train["user_id"].map(user_index).to_numpy(),
            train["tv_show_id"].map(item_index).to_numpy(),
        ),
    ),
    shape=(len(train_users), len(item_ids)),
)

profiles = normalize(
    (history @ item_vectors).tocsr(),
    norm="l2",
    axis=1,
).tocsr()

# Check normalization for nonzero item vectors.
item_norms = np.sqrt(
    item_vectors.multiply(item_vectors).sum(axis=1)
).A1
assert np.allclose(item_norms[item_norms > 0], 1.0)

candidate_set = set(candidate_ids)
per_user_rows = []
recommendation_rows = []

print(
    f"Train metadata items: {len(item_ids):,} | "
    f"Features: {item_vectors.shape[1]:,} | "
    f"Cohort: {len(cohort):,}",
    flush=True,
)

for number, user in enumerate(cohort, start=1):
    has_history = user in user_index
    zero_profile = True
    scores = np.zeros(len(candidate_ids), dtype=np.float64)

    if has_history:
        profile = profiles.getrow(user_index[user])
        zero_profile = profile.nnz == 0

        if not zero_profile:
            scores = (
                candidate_vectors @ profile.T
            ).toarray().ravel()

    assert np.isfinite(scores).all()
    assert (scores >= -1e-12).all()

    # Rank positive content similarities.
    # Ties are broken by numeric programme ID.
    positive_indices = np.flatnonzero(scores > 0)
    positive_indices = sorted(
        positive_indices,
        key=lambda index: (
            -float(scores[index]),
            int(candidate_ids[index]),
        ),
    )

    selected = [
        candidate_ids[index]
        for index in positive_indices[:5]
    ]
    personalized_count = len(selected)

    # Same train-popularity fallback as the baseline.
    for item in popularity_order:
        if len(selected) == 5:
            break
        if item not in selected:
            selected.append(item)

    assert len(selected) == len(set(selected)) == 5
    assert set(selected).issubset(candidate_set)

    if not has_history or zero_profile:
        assert selected == popularity_top5

    score_by_item = {
        candidate_ids[index]: float(scores[index])
        for index in positive_indices[:5]
    }

    per_user_rows.append(
        {
            "user_id": user,
            "has_train_history": has_history,
            "zero_content_profile": zero_profile,
            "positive_content_slots": personalized_count,
            "fallback_slots": 5 - personalized_count,
            "ap_at_5": ap5(targets[user], selected),
            "recommended_ids": " ".join(selected),
        }
    )

    for rank, item in enumerate(selected, start=1):
        is_content = rank <= personalized_count
        recommendation_rows.append(
            {
                "user_id": user,
                "method": "content_based",
                "rank": rank,
                "tv_show_id": item,
                "content_score": score_by_item.get(item, 0.0),
                "train_popularity": int(popularity.get(item, 0)),
                "source": (
                    "content_similarity"
                    if is_content else "popularity_fallback"
                ),
            }
        )

    if number % 250 == 0 or number == len(cohort):
        print(
            f"Users evaluated: {number}/{len(cohort)}",
            flush=True,
        )

per_user = pd.DataFrame(per_user_rows)
recommendations = pd.DataFrame(recommendation_rows)

warm = per_user.loc[per_user["has_train_history"]]
cold = per_user.loc[~per_user["has_train_history"]]

with (WORK / "popularity_validation_summary.json").open(
    encoding="utf-8"
) as handle:
    baseline = json.load(handle)

assert len(cohort) == baseline["cohort_size"]
assert len(cold) == baseline["cold_users"]

summary = {
    "method": "content_based",
    "metric": "MAP@5",
    "cohort_size": len(cohort),
    "warm_users": len(warm),
    "cold_users": len(cold),
    "map_at_5": float(per_user["ap_at_5"].mean()),
    "warm_map_at_5": (
        float(warm["ap_at_5"].mean()) if len(warm) else None
    ),
    "cold_map_at_5": (
        float(cold["ap_at_5"].mean()) if len(cold) else None
    ),
    "popularity_map_at_5": baseline["map_at_5"],
    "difference_from_popularity": (
        float(per_user["ap_at_5"].mean())
        - baseline["map_at_5"]
    ),
    "settings": {
        "field_weights": FIELD_WEIGHTS,
        "item_weighting": "Smoothed train-only IDF, then field weights",
        "item_normalization": "L2",
        "history_weighting": "log1p(train watch_count)",
        "profile_normalization": "L2",
        "ranking": "Positive cosine similarity, ID ascending for ties",
        "fallback": "Train-popularity restricted to candidates",
        "parameter_selection": "Fixed initial settings; no sweep yet",
        "randomness": "No random sampling or initialization",
        "repeats_allowed": True,
    },
    "train_metadata_items": len(item_ids),
    "features": int(item_vectors.shape[1]),
    "validation_candidates": len(candidate_ids),
    "candidates_absent_from_train_metadata": sum(
        item not in item_index for item in candidate_ids
    ),
    "users_with_zero_content_profile": int(
        per_user["zero_content_profile"].sum()
    ),
    "users_requiring_any_fallback": int(
        per_user["fallback_slots"].gt(0).sum()
    ),
    "mean_positive_content_slots": float(
        per_user["positive_content_slots"].mean()
    ),
    "sklearn_version": sklearn.__version__,
    "runtime_seconds": round(time.perf_counter() - started, 3),
}

per_user.to_csv(
    WORK / "content_validation_per_user.csv", index=False
)
recommendations.to_csv(
    WORK / "content_validation_recommendations.csv",
    index=False,
)

with (WORK / "content_validation_summary.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(
        summary, handle,
        ensure_ascii=False, indent=2, allow_nan=False,
    )

joblib.dump(
    {
        "vectorizer": vectorizer,
        "tfidf": tfidf,
        "feature_weights": feature_weights,
        "field_weights": FIELD_WEIGHTS,
        "item_ids": item_ids,
        "item_vectors": item_vectors,
    },
    WORK / "content_train_model.joblib",
)

print("\nCONTENT VALIDATION COMPLETE")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved:")
print("  content_validation_per_user.csv")
print("  content_validation_recommendations.csv")
print("  content_validation_summary.json")
print("  content_train_model.joblib")
print("\nPPR and Kaggle submission have NOT been run.")

CONTENT-BASED — VALIDATION
Train metadata items: 5,834 | Features: 12,055 | Cohort: 1,238
Users evaluated: 250/1238
Users evaluated: 500/1238
Users evaluated: 750/1238
Users evaluated: 1000/1238
Users evaluated: 1238/1238

CONTENT VALIDATION COMPLETE
{
  "method": "content_based",
  "metric": "MAP@5",
  "cohort_size": 1238,
  "warm_users": 1224,
  "cold_users": 14,
  "map_at_5": 0.1623815293484114,
  "warm_map_at_5": 0.16235974945533768,
  "cold_map_at_5": 0.1642857142857143,
  "popularity_map_at_5": 0.2961265930712619,
  "difference_from_popularity": -0.13374506372285047,
  "settings": {
    "field_weights": {
      "category": 2.0,
      "genre": 3.0,
      "year": 0.5,
      "director": 1.0,
      "actor": 0.5
    },
    "item_weighting": "Smoothed train-only IDF, then field weights",
    "item_normalization": "L2",
    "history_weighting": "log1p(train watch_count)",
    "profile_normalization": "L2",
    "ranking": "Positive cosine similarity, ID ascending for ties",
    "fallback

In [14]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
from scipy.sparse import csr_matrix, diags

WORK = Path("/kaggle/working")

ALPHAS = [0.50, 0.70, 0.85, 0.95]
TOP_NEIGHBORS = 50
BATCH_SIZE = 64
TOLERANCE = 1e-8
MAX_ITERATIONS = 500

started = time.perf_counter()


def ap5(relevant, predictions):
    relevant = set(relevant)
    if not relevant:
        raise ValueError("Empty target.")

    hits = 0
    total = 0.0
    seen = set()

    for rank, item in enumerate(predictions[:5], start=1):
        if item in seen:
            continue
        seen.add(item)
        if item in relevant:
            hits += 1
            total += hits / rank

    return total / min(5, len(relevant))


def propagate(transition, dangling, restart, alpha):
    """
    Each restart column describes one user.
    Return distributions, batch iteration count, per-user residuals.
    """
    p = restart.copy()

    for iteration in range(1, MAX_ITERATIONS + 1):
        dangling_mass = p[dangling].sum(axis=0)

        next_p = (
            alpha * (transition.T @ p)
            + restart * (
                (1.0 - alpha) + alpha * dangling_mass
            )[None, :]
        )

        residuals = np.abs(next_p - p).sum(axis=0)
        p = next_p

        if float(residuals.max()) < TOLERANCE:
            break

    assert np.isfinite(p).all()
    assert p.min() >= -1e-12
    assert np.allclose(p.sum(axis=0), 1.0, atol=1e-8)

    return p, iteration, residuals


# Check orientation and dangling correction against a dense solve.
test_q = csr_matrix(
    [
        [0.0, 1.0, 0.0],
        [0.5, 0.0, 0.5],
        [0.0, 0.0, 0.0],
    ]
)
test_dangling = np.array([False, False, True])

for test_v in [
    np.array([1.0, 0.0, 0.0]),
    np.array([0.0, 0.0, 1.0]),
]:
    actual, _, residual = propagate(
        test_q, test_dangling, test_v[:, None], 0.85
    )

    completed_transition = (
        test_q.toarray()
        + np.outer(test_dangling.astype(float), test_v)
    )
    expected = np.linalg.solve(
        np.eye(3) - 0.85 * completed_transition.T,
        0.15 * test_v,
    )

    assert np.allclose(actual[:, 0], expected, atol=1e-7)
    assert residual[0] < TOLERANCE

print("PASS: PPR orientation, mass and dangling-node checks")

train = pd.read_csv(
    WORK / "train_interactions.csv",
    dtype={"user_id": str, "tv_show_id": str},
)

cohort = pd.read_csv(
    WORK / "validation_cohort.csv",
    dtype=str,
)["user_id"].tolist()

target_frame = pd.read_csv(
    WORK / "validation_targets.csv",
    dtype=str,
    keep_default_na=False,
)
targets = {
    row.user_id: row.relevant_ids.split()
    for row in target_frame.itertuples(index=False)
}

candidates = sorted(
    pd.read_csv(
        WORK / "validation_candidate_ids.csv",
        dtype=str,
    )["tv_show_id"].tolist(),
    key=int,
)
candidate_set = set(candidates)

assert len(candidates) >= 5
assert len(candidates) == len(candidate_set)
assert "0" not in candidate_set
assert all(targets.get(user) for user in cohort)

users = sorted(train["user_id"].unique(), key=int)
items = sorted(train["tv_show_id"].unique(), key=int)
user_index = {user: index for index, user in enumerate(users)}
item_index = {item: index for index, item in enumerate(items)}

row_indices = train["user_id"].map(user_index).to_numpy()
col_indices = train["tv_show_id"].map(item_index).to_numpy()

# Binary co-view counts: distinct train users shared by two items.
binary_history = csr_matrix(
    (
        np.ones(len(train), dtype=np.float64),
        (row_indices, col_indices),
    ),
    shape=(len(users), len(items)),
)
assert binary_history.max() == 1

print(
    f"Building graph: {len(items):,} programme nodes",
    flush=True,
)

cooccurrence = (binary_history.T @ binary_history).tocsr()
cooccurrence.setdiag(0)
cooccurrence.eliminate_zeros()

item_user_counts = np.asarray(
    binary_history.sum(axis=0)
).ravel()

inverse_sqrt = 1.0 / np.sqrt(item_user_counts)

# Cosine-normalized binary co-view similarity.
similarity = (
    diags(inverse_sqrt)
    @ cooccurrence
    @ diags(inverse_sqrt)
).tocsr()

del cooccurrence

# Keep the 50 strongest outgoing neighbors per node.
# Node order is numeric ID order, giving deterministic ties.
graph_rows = []
graph_cols = []
graph_values = []

for row in range(len(items)):
    start = similarity.indptr[row]
    stop = similarity.indptr[row + 1]

    columns = similarity.indices[start:stop]
    values = similarity.data[start:stop]

    order = np.lexsort((columns, -values))[:TOP_NEIGHBORS]

    graph_rows.extend([row] * len(order))
    graph_cols.extend(columns[order].tolist())
    graph_values.extend(values[order].tolist())

graph = csr_matrix(
    (graph_values, (graph_rows, graph_cols)),
    shape=(len(items), len(items)),
)

# Symmetric union: retain an edge if either endpoint selected it.
graph = graph.maximum(graph.T).tocsr()
graph.setdiag(0)
graph.eliminate_zeros()

del similarity, graph_rows, graph_cols, graph_values

degrees = np.asarray(graph.sum(axis=1)).ravel()
dangling = degrees == 0
inverse_degree = np.zeros_like(degrees)
inverse_degree[~dangling] = 1.0 / degrees[~dangling]

transition = (diags(inverse_degree) @ graph).tocsr()

row_sums = np.asarray(transition.sum(axis=1)).ravel()
assert np.allclose(row_sums[~dangling], 1.0)
assert np.allclose(row_sums[dangling], 0.0)
assert transition.data.min(initial=0.0) >= 0.0

# Restart uses train counts, with log damping.
weighted_history = csr_matrix(
    (
        np.log1p(
            train["watch_count"].to_numpy(dtype=np.float64)
        ),
        (row_indices, col_indices),
    ),
    shape=binary_history.shape,
)

popularity = (
    train.groupby("tv_show_id")["watch_count"].sum().to_dict()
)
popularity_order = sorted(
    candidates,
    key=lambda item: (-int(popularity.get(item, 0)), int(item)),
)
popularity_top5 = popularity_order[:5]

# Items not in the train graph have no graph score.
eligible_node_indices = np.array(
    [
        index for index, item in enumerate(items)
        if item in candidate_set
    ],
    dtype=np.int64,
)

warm_users = [user for user in cohort if user in user_index]
cold_users = [user for user in cohort if user not in user_index]

graph_summary = {
    "nodes": len(items),
    "undirected_edges": int(graph.nnz // 2),
    "dangling_nodes": int(dangling.sum()),
    "top_neighbors_before_symmetric_union": TOP_NEIGHBORS,
    "edge_weight": (
        "Shared distinct train users divided by "
        "sqrt(item user count product)"
    ),
    "node_universe": "Programmes watched in train",
    "validation_candidates_absent_from_graph": sum(
        item not in item_index for item in candidates
    ),
}

print(json.dumps(graph_summary, indent=2), flush=True)

all_results = []

for alpha in ALPHAS:
    alpha_started = time.perf_counter()
    per_user_rows = []
    recommendation_rows = []

    for offset in range(0, len(warm_users), BATCH_SIZE):
        batch = warm_users[offset:offset + BATCH_SIZE]
        history_rows = [user_index[user] for user in batch]

        restart = weighted_history[history_rows].T.toarray()
        restart /= restart.sum(axis=0, keepdims=True)

        probabilities, iterations, residuals = propagate(
            transition, dangling, restart, alpha
        )

        for column, user in enumerate(batch):
            node_scores = probabilities[:, column]

            positive = eligible_node_indices[
                node_scores[eligible_node_indices] > 0
            ]
            order = np.lexsort(
                (positive, -node_scores[positive])
            )
            selected_nodes = positive[order[:5]]
            selected = [items[index] for index in selected_nodes]
            personalized_count = len(selected)

            for item in popularity_order:
                if len(selected) == 5:
                    break
                if item not in selected:
                    selected.append(item)

            assert len(selected) == len(set(selected)) == 5
            assert set(selected).issubset(candidate_set)

            score_lookup = {
                items[index]: float(node_scores[index])
                for index in selected_nodes
            }

            per_user_rows.append(
                {
                    "user_id": user,
                    "alpha": alpha,
                    "has_train_history": True,
                    "ap_at_5": ap5(targets[user], selected),
                    "iterations": iterations,
                    "residual": float(residuals[column]),
                    "converged": bool(
                        residuals[column] < TOLERANCE
                    ),
                    "fallback_slots": 5 - personalized_count,
                    "recommended_ids": " ".join(selected),
                }
            )

            for rank, item in enumerate(selected, start=1):
                recommendation_rows.append(
                    {
                        "user_id": user,
                        "method": "ppr",
                        "alpha": alpha,
                        "rank": rank,
                        "tv_show_id": item,
                        "score": score_lookup.get(item, 0.0),
                        "source": (
                            "ppr"
                            if rank <= personalized_count
                            else "popularity_fallback"
                        ),
                    }
                )

        print(
            f"alpha={alpha:.2f} | "
            f"warm users {min(offset + BATCH_SIZE, len(warm_users))}"
            f"/{len(warm_users)}",
            flush=True,
        )

    for user in cold_users:
        per_user_rows.append(
            {
                "user_id": user,
                "alpha": alpha,
                "has_train_history": False,
                "ap_at_5": ap5(targets[user], popularity_top5),
                "iterations": 0,
                "residual": None,
                "converged": None,
                "fallback_slots": 5,
                "recommended_ids": " ".join(popularity_top5),
            }
        )

        for rank, item in enumerate(popularity_top5, start=1):
            recommendation_rows.append(
                {
                    "user_id": user,
                    "method": "ppr",
                    "alpha": alpha,
                    "rank": rank,
                    "tv_show_id": item,
                    "score": 0.0,
                    "source": "popularity_fallback",
                }
            )

    per_user = pd.DataFrame(per_user_rows)
    recs = pd.DataFrame(recommendation_rows)

    assert set(per_user["user_id"]) == set(cohort)
    assert per_user["user_id"].is_unique

    warm = per_user.loc[per_user["has_train_history"]]
    cold = per_user.loc[~per_user["has_train_history"]]

    result = {
        "alpha": alpha,
        "cohort_size": len(per_user),
        "map_at_5": float(per_user["ap_at_5"].mean()),
        "warm_map_at_5": (
            float(warm["ap_at_5"].mean()) if len(warm) else None
        ),
        "cold_map_at_5": (
            float(cold["ap_at_5"].mean()) if len(cold) else None
        ),
        "nonconverged_warm_users": int(
            warm["converged"].eq(False).sum()
        ),
        "max_residual": (
            float(warm["residual"].max()) if len(warm) else None
        ),
        "max_batch_iterations": (
            int(warm["iterations"].max()) if len(warm) else 0
        ),
        "runtime_seconds": round(
            time.perf_counter() - alpha_started, 3
        ),
    }

    tag = f"{alpha:.2f}"
    per_user.to_csv(
        WORK / f"ppr_alpha_{tag}_per_user.csv", index=False
    )
    recs.to_csv(
        WORK / f"ppr_alpha_{tag}_recommendations.csv", index=False
    )
    all_results.append(result)

    print("\nALPHA RESULT")
    print(json.dumps(result, indent=2), flush=True)

# Prefer smaller alpha for an exact metric tie.
eligible_results = [
    result for result in all_results
    if result["nonconverged_warm_users"] == 0
]
if not eligible_results:
    raise RuntimeError(
        "No fully converged alpha setting. "
        "Inspect saved convergence results before selection."
    )

best = sorted(
    eligible_results,
    key=lambda result: (-result["map_at_5"], result["alpha"]),
)[0]

selected_alpha = best["alpha"]
tag = f"{selected_alpha:.2f}"

pd.read_csv(
    WORK / f"ppr_alpha_{tag}_per_user.csv",
    dtype={"user_id": str},
).to_csv(
    WORK / "ppr_validation_per_user.csv", index=False
)

pd.read_csv(
    WORK / f"ppr_alpha_{tag}_recommendations.csv",
    dtype={"user_id": str, "tv_show_id": str},
).to_csv(
    WORK / "ppr_validation_recommendations.csv", index=False
)

pd.DataFrame(all_results).to_csv(
    WORK / "ppr_alpha_sweep.csv", index=False
)

summary = {
    "method": "ppr",
    "selected_alpha": selected_alpha,
    "selection": "Validation MAP@5; smaller alpha for exact ties",
    "cohort_size": len(cohort),
    "warm_users": len(warm_users),
    "cold_users": len(cold_users),
    "map_at_5": best["map_at_5"],
    "warm_map_at_5": best["warm_map_at_5"],
    "cold_map_at_5": best["cold_map_at_5"],
    "graph": graph_summary,
    "restart": "Normalized log1p(train watch_count)",
    "dangling_policy": "Redistribute mass to user restart vector",
    "tolerance": TOLERANCE,
    "max_iterations": MAX_ITERATIONS,
    "batch_size": BATCH_SIZE,
    "iteration_reporting": (
        "Users in a batch share its stopping iteration; "
        "residuals are measured individually"
    ),
    "repeats_allowed": True,
    "fallback": "Shared train-popularity fallback",
    "scipy_version": scipy.__version__,
    "alpha_results": all_results,
    "runtime_seconds": round(time.perf_counter() - started, 3),
}

with (WORK / "ppr_validation_summary.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(
        summary, handle, indent=2,
        ensure_ascii=False, allow_nan=False,
    )

joblib.dump(
    {
        "item_ids": items,
        "transition": transition,
        "dangling": dangling,
        "selected_alpha": selected_alpha,
        "top_neighbors": TOP_NEIGHBORS,
        "restart_weighting": "log1p(watch_count)",
    },
    WORK / "ppr_train_model.joblib",
)

print("\nPPR VALIDATION COMPLETE")
print(
    f"Selected alpha: {selected_alpha:.2f}\n"
    f"MAP@5: {best['map_at_5']:.10f}\n"
    f"Total runtime: {summary['runtime_seconds']} seconds"
)
print("\nSaved:")
print("  ppr_alpha_sweep.csv")
print("  ppr_validation_per_user.csv")
print("  ppr_validation_recommendations.csv")
print("  ppr_validation_summary.json")
print("  ppr_train_model.joblib")
print("  Per-alpha recommendation and convergence CSV files")
print("\nNo Kaggle submission has been sent.")

PASS: PPR orientation, mass and dangling-node checks
Building graph: 2,489 programme nodes
{
  "nodes": 2489,
  "undirected_edges": 81292,
  "dangling_nodes": 0,
  "top_neighbors_before_symmetric_union": 50,
  "edge_weight": "Shared distinct train users divided by sqrt(item user count product)",
  "node_universe": "Programmes watched in train",
  "validation_candidates_absent_from_graph": 4498
}
alpha=0.50 | warm users 64/1224
alpha=0.50 | warm users 128/1224
alpha=0.50 | warm users 192/1224
alpha=0.50 | warm users 256/1224
alpha=0.50 | warm users 320/1224
alpha=0.50 | warm users 384/1224
alpha=0.50 | warm users 448/1224
alpha=0.50 | warm users 512/1224
alpha=0.50 | warm users 576/1224
alpha=0.50 | warm users 640/1224
alpha=0.50 | warm users 704/1224
alpha=0.50 | warm users 768/1224
alpha=0.50 | warm users 832/1224
alpha=0.50 | warm users 896/1224
alpha=0.50 | warm users 960/1224
alpha=0.50 | warm users 1024/1224
alpha=0.50 | warm users 1088/1224
alpha=0.50 | warm users 1152/1224
alpha

In [16]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

WORK = Path("/kaggle/working")

train = pd.read_csv(
    WORK / "train_interactions.csv",
    dtype={"user_id": str, "tv_show_id": str},
)

validation = pd.read_csv(
    WORK / "validation_interactions.csv",
    dtype={"user_id": str, "tv_show_id": str},
)

cohort = pd.read_csv(
    WORK / "validation_cohort.csv",
    dtype=str,
)["user_id"].tolist()

target_frame = pd.read_csv(
    WORK / "validation_targets.csv",
    dtype=str,
    keep_default_na=False,
)
targets = {
    row.user_id: row.relevant_ids.split()
    for row in target_frame.itertuples(index=False)
}

train_metadata = pd.read_csv(
    WORK / "train_item_metadata.csv",
    dtype=str,
    keep_default_na=False,
)
validation_metadata = pd.read_csv(
    WORK / "validation_item_metadata.csv",
    dtype=str,
    keep_default_na=False,
)

# Metadata are used here only for human-readable reporting.
# This cell does not fit or update recommendation models.
metadata = pd.concat(
    [train_metadata, validation_metadata],
    ignore_index=True,
).drop_duplicates("tv_show_id", keep="first")

lookup = metadata.set_index("tv_show_id").to_dict("index")


def describe_item(item):
    record = lookup.get(str(item), {})
    return {
        "title": record.get("tv_show_title", "[title unavailable]"),
        "category": record.get(
            "tv_show_category", "[category unavailable]"
        ),
    }


def ap5(relevant, predictions):
    relevant = set(relevant)
    hits = 0
    total = 0.0
    seen = set()

    for rank, item in enumerate(predictions[:5], start=1):
        if item in seen:
            continue
        seen.add(item)
        if item in relevant:
            hits += 1
            total += hits / rank

    return total / min(5, len(relevant))


# Select users using train activity only.
activity = (
    train.groupby("user_id")["watch_count"]
    .agg(["sum", "size"])
    .rename(
        columns={
            "sum": "train_airing_views",
            "size": "train_unique_programmes",
        }
    )
    .reset_index()
)

activity = activity.loc[
    activity["user_id"].isin(cohort)
].copy()
activity["_numeric_user_id"] = activity["user_id"].map(int)

if len(activity) < 2:
    raise ValueError("Need at least two warm validation users.")

heavy = activity.sort_values(
    ["train_airing_views", "_numeric_user_id"],
    ascending=[False, True],
    kind="stable",
).iloc[0]

light = activity.loc[
    activity["user_id"].ne(heavy["user_id"])
].sort_values(
    ["train_airing_views", "_numeric_user_id"],
    ascending=[True, True],
    kind="stable",
).iloc[0]

selected_users = [
    ("heavy", str(heavy["user_id"])),
    ("light", str(light["user_id"])),
]

recommendation_files = {
    "popularity": "popularity_validation_recommendations.csv",
    "content_based": "content_validation_recommendations.csv",
    "ppr": "ppr_validation_recommendations.csv",
}

recommendation_tables = {
    method: pd.read_csv(
        WORK / filename,
        dtype={"user_id": str, "tv_show_id": str},
    )
    for method, filename in recommendation_files.items()
}

summary_rows = []
history_rows = []
actual_rows = []
example_rows = []

train_popularity = (
    train.groupby("tv_show_id")["watch_count"]
    .sum()
    .to_dict()
)

for viewer_type, user in selected_users:
    user_history = train.loc[train["user_id"].eq(user)].copy()
    user_history["_item_id"] = user_history["tv_show_id"].map(int)
    user_history = user_history.sort_values(
        ["watch_count", "_item_id"],
        ascending=[False, True],
        kind="stable",
    )

    user_actual = validation.loc[
        validation["user_id"].eq(user)
    ].copy()
    user_actual["_item_id"] = user_actual["tv_show_id"].map(int)
    user_actual = user_actual.sort_values(
        ["watch_count", "_item_id"],
        ascending=[False, True],
        kind="stable",
    )

    target_set = set(targets[user])

    print("\n" + "=" * 90)
    print(f"{viewer_type.upper()} VIEWER: {user}")
    print(
        f"Train airing views: {int(user_history['watch_count'].sum())}"
        f" | Unique train programmes: {len(user_history)}"
        f" | Validation programmes: {len(user_actual)}"
    )

    summary_rows.append(
        {
            "viewer_type": viewer_type,
            "user_id": user,
            "train_airing_views": int(
                user_history["watch_count"].sum()
            ),
            "train_unique_programmes": len(user_history),
            "validation_unique_programmes": len(user_actual),
            "validation_target_ids": " ".join(targets[user]),
        }
    )

    # Save full history, print only its ten most frequent programmes.
    for row in user_history.itertuples(index=False):
        history_rows.append(
            {
                "viewer_type": viewer_type,
                "user_id": user,
                "tv_show_id": row.tv_show_id,
                "train_watch_count": int(row.watch_count),
                **describe_item(row.tv_show_id),
            }
        )

    print("\nTRAIN HISTORY — TOP 10")
    print(
        pd.DataFrame(
            [row for row in history_rows if row["user_id"] == user]
        ).head(10)[
            ["tv_show_id", "train_watch_count", "title", "category"]
        ].to_string(index=False)
    )

    for row in user_actual.itertuples(index=False):
        actual_rows.append(
            {
                "viewer_type": viewer_type,
                "user_id": user,
                "tv_show_id": row.tv_show_id,
                "validation_watch_count": int(row.watch_count),
                "in_validation_top5_target": (
                    row.tv_show_id in target_set
                ),
                **describe_item(row.tv_show_id),
            }
        )

    print("\nACTUAL VALIDATION TOP-5 TARGET")
    print(
        pd.DataFrame(
            [
                row for row in actual_rows
                if row["user_id"] == user
                and row["in_validation_top5_target"]
            ]
        )[
            [
                "tv_show_id",
                "validation_watch_count",
                "title",
                "category",
            ]
        ].to_string(index=False)
    )

    for method, frame in recommendation_tables.items():
        recommendations = frame.loc[
            frame["user_id"].eq(user)
        ].sort_values("rank")

        assert len(recommendations) == 5
        assert recommendations["tv_show_id"].is_unique
        assert recommendations["rank"].tolist() == [1, 2, 3, 4, 5]

        ids = recommendations["tv_show_id"].tolist()
        score = ap5(targets[user], ids)

        actual_counts = dict(
            zip(
                user_actual["tv_show_id"],
                user_actual["watch_count"],
            )
        )
        history_ids = set(user_history["tv_show_id"])

        method_rows = []

        for row in recommendations.itertuples(index=False):
            item = row.tv_show_id
            record = {
                "viewer_type": viewer_type,
                "user_id": user,
                "method": method,
                "rank": int(row.rank),
                "tv_show_id": item,
                **describe_item(item),
                "in_train_history": item in history_ids,
                "in_validation_top5_target": item in target_set,
                "validation_watch_count": int(
                    actual_counts.get(item, 0)
                ),
                "train_popularity": int(
                    train_popularity.get(item, 0)
                ),
                "user_ap_at_5": float(score),
            }
            example_rows.append(record)
            method_rows.append(record)

        print(f"\n{method.upper()} — AP@5 = {score:.6f}")
        print(
            pd.DataFrame(method_rows)[
                [
                    "rank", "tv_show_id", "title", "category",
                    "in_train_history",
                    "in_validation_top5_target",
                    "validation_watch_count",
                ]
            ].to_string(index=False)
        )

pd.DataFrame(summary_rows).to_csv(
    WORK / "example_users.csv", index=False
)
pd.DataFrame(history_rows).to_csv(
    WORK / "example_users_train_history.csv", index=False
)
pd.DataFrame(actual_rows).to_csv(
    WORK / "example_users_validation_actual.csv", index=False
)
pd.DataFrame(example_rows).to_csv(
    WORK / "example_users_recommendations.csv", index=False
)

method_comparison = []

for method in ["popularity", "content", "ppr"]:
    with (WORK / f"{method}_validation_summary.json").open(
        encoding="utf-8"
    ) as handle:
        result = json.load(handle)

    method_comparison.append(
        {
            "method": result["method"],
            "cohort_size": result["cohort_size"],
            "map_at_5": result["map_at_5"],
            "warm_map_at_5": result["warm_map_at_5"],
            "cold_map_at_5": result["cold_map_at_5"],
            "stage_runtime_seconds": result["runtime_seconds"],
            "selected_alpha": result.get("selected_alpha"),
        }
    )

comparison = pd.DataFrame(method_comparison)
comparison.to_csv(
    WORK / "validation_method_comparison.csv", index=False
)

print("\n" + "=" * 90)
print("METHOD COMPARISON")
print(comparison.to_string(index=False))
print(
    "\nRuntime columns describe different stages:"
    " PPR includes graph construction and the four-alpha sweep."
    " They are not pure per-user latency comparisons."
)

print("\nUSER EXAMPLES COMPLETE")
print("Saved:")
print("  example_users.csv")
print("  example_users_train_history.csv")
print("  example_users_validation_actual.csv")
print("  example_users_recommendations.csv")
print("  validation_method_comparison.csv")
print("\nNo final retraining or Kaggle submission was performed.")


HEAVY VIEWER: 10500740270951698446
Train airing views: 904 | Unique train programmes: 97 | Validation programmes: 123

TRAIN HISTORY — TOP 10
tv_show_id  train_watch_count                                                     title    category
  12001682                401                         М/с 'Губка Боб Квадратные Штаны'.       Детям
  12002856                 68                             М/с 'Элвиннн!!! и бурундуки'.       Детям
   2400480                 62                                              ТСН. Неделя.        Инфо
  12001697                 54                           М/с 'Аватар: Легенда об Аанге'.       Детям
  12001734                 54   М/с 'Волшебные покровители', 161 с. 'Женат на мамочке'.       Детям
    240081                 32                                              Факты. День.        Инфо
  12002771                 22                                   М/с 'Черепашки-ниндзя'.       Детям
     20088                 18 'Завтрак с 1+1'. Информацио

In [17]:
import gc
import hashlib
import json
import time
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags

BASE = Path(
    "/kaggle/input/competitions/sweettv-tv-program-recommender"
)
WORK = Path("/kaggle/working")

FINAL_ALPHA = 0.70
TOP_NEIGHBORS = 50
BATCH_SIZE = 64
TOLERANCE = 1e-8
MAX_ITERATIONS = 500

FINAL_START = int(
    pd.Timestamp("2020-03-09").value // 1_000_000_000
)
FINAL_END = int(
    pd.Timestamp("2020-07-27").value // 1_000_000_000
)

started = time.perf_counter()

# Reuse the corrected and tested interval-join functions.
for function_name in ["build_counts", "load_views"]:
    if not callable(globals().get(function_name)):
        raise RuntimeError(
            f"Missing function {function_name}. "
            "Run this cell in the same notebook session "
            "as the successful interaction build."
        )

with (WORK / "ppr_validation_summary.json").open(
    encoding="utf-8"
) as handle:
    selected_result = json.load(handle)

assert selected_result["selected_alpha"] == FINAL_ALPHA
assert selected_result["graph"][
    "top_neighbors_before_symmetric_union"
] == TOP_NEIGHBORS

print("FINAL TRAINING AND SUBMISSION BUILD", flush=True)
print(f"Fixed alpha: {FINAL_ALPHA}", flush=True)

# ------------------------------------------------------------------
# 1. Build a schedule for the complete final training window.
# ------------------------------------------------------------------

metadata_columns = [
    "tv_show_title", "tv_show_category",
    "tv_show_genre_1", "tv_show_genre_2", "tv_show_genre_3",
    "year_of_production", "director", "actors",
]

schedule_parts = []
schedule_input_stats = []

for filename in [
    "export_arh_11-20-final.csv",
    "export_arh_21-30-final.csv",
]:
    stats = {
        "file": filename,
        "records": 0,
        "excluded_nonpositive_duration": 0,
        "excluded_zero_id_after_duration_filter": 0,
    }

    for chunk in pd.read_csv(
        BASE / filename,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        chunksize=100_000,
    ):
        stats["records"] += len(chunk)

        duration = pd.to_numeric(
            chunk["duration"], errors="raise"
        )
        assert np.isfinite(duration).all()
        assert duration.eq(np.floor(duration)).all()

        positive = duration.gt(0)
        nonzero = chunk["tv_show_id"].ne("0")

        stats["excluded_nonpositive_duration"] += int(
            (~positive).sum()
        )
        stats["excluded_zero_id_after_duration_filter"] += int(
            (positive & ~nonzero).sum()
        )

        selected = chunk.loc[positive & nonzero].copy()

        if selected.empty:
            continue

        parsed_start = pd.to_datetime(
            selected["start_time"],
            format="%d.%m.%Y %H:%M:%S",
            errors="raise",
        )

        selected["start"] = (
            parsed_start.astype("int64") // 1_000_000_000
        )
        selected["stop"] = (
            selected["start"]
            + duration.loc[selected.index].astype("int64")
        )
        selected["channel"] = selected["channel_id"]
        selected["item"] = selected["tv_show_id"]
        selected["source"] = filename

        schedule_parts.append(
            selected[
                ["channel", "item", "start", "stop", "source"]
                + metadata_columns
            ]
        )

    schedule_input_stats.append(stats)
    print(f"Loaded: {filename}", flush=True)

schedule = pd.concat(schedule_parts, ignore_index=True)
del schedule_parts
gc.collect()

identity = ["channel", "start", "stop", "item"]
duplicates_removed = int(
    schedule.duplicated(identity).sum()
)
schedule = schedule.drop_duplicates(identity).copy()

conflicts = schedule.duplicated(
    ["channel", "start"], keep=False
)

if conflicts.any():
    schedule.loc[conflicts].to_csv(
        WORK / "final_schedule_conflicts.csv", index=False
    )
    raise ValueError(
        "Conflicting schedule records detected. "
        "Saved final_schedule_conflicts.csv."
    )

contained = (
    schedule["start"].ge(FINAL_START)
    & schedule["stop"].le(FINAL_END)
)
overlapping = (
    schedule["start"].lt(FINAL_END)
    & schedule["stop"].gt(FINAL_START)
)

schedule_stats = {
    "sources": schedule_input_stats,
    "duplicates_removed": duplicates_removed,
    "boundary_crossing_excluded": int(
        (overlapping & ~contained).sum()
    ),
    "entirely_outside": int((~overlapping).sum()),
    "retained_airings": int(contained.sum()),
}

schedule = schedule.loc[contained].copy()

print("\nFinal train schedule:")
print(json.dumps(schedule_stats, indent=2), flush=True)

# ------------------------------------------------------------------
# 2. Rebuild interactions for one continuous training window.
# ------------------------------------------------------------------

views, viewing_stats = load_views(
    set(schedule["channel"].unique())
)

print("\nBuilding final interactions...", flush=True)

final_counts = build_counts(
    views,
    schedule,
    (FINAL_START, FINAL_END),
    progress=True,
)

del views
gc.collect()

final_interactions = pd.DataFrame(
    [
        {
            "user_id": user,
            "tv_show_id": item,
            "watch_count": int(count),
        }
        for (user, item), count in sorted(
            final_counts.items(),
            key=lambda entry: (
                int(entry[0][0]), int(entry[0][1])
            ),
        )
    ]
)

del final_counts

if final_interactions.empty:
    raise ValueError("No final training interactions.")

assert final_interactions["watch_count"].gt(0).all()
assert not final_interactions.duplicated(
    ["user_id", "tv_show_id"]
).any()

final_interactions.to_csv(
    WORK / "final_train_interactions.csv", index=False
)

final_metadata = (
    schedule.sort_values(
        ["item", "start", "channel", "source"],
        kind="stable",
    )
    .drop_duplicates("item")
    [["item"] + metadata_columns]
    .rename(columns={"item": "tv_show_id"})
)
final_metadata.to_csv(
    WORK / "final_train_item_metadata.csv", index=False
)

final_interaction_stats = {
    "user_programme_pairs": len(final_interactions),
    "users": int(final_interactions["user_id"].nunique()),
    "programmes": int(
        final_interactions["tv_show_id"].nunique()
    ),
    "qualified_airing_views": int(
        final_interactions["watch_count"].sum()
    ),
}
print("\nFinal interaction statistics:")
print(json.dumps(final_interaction_stats, indent=2), flush=True)

del schedule
gc.collect()

# ------------------------------------------------------------------
# 3. Read future schedule only to establish candidate availability.
# No test-period viewing behaviour is read.
# ------------------------------------------------------------------

candidate_set = set()

for chunk in pd.read_csv(
    BASE / "export_arh_31-42-final.csv",
    usecols=["tv_show_id", "duration"],
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
    chunksize=100_000,
):
    duration = pd.to_numeric(
        chunk["duration"], errors="raise"
    )
    assert np.isfinite(duration).all()

    valid = (
        duration.gt(0)
        & chunk["tv_show_id"].ne("0")
        & chunk["tv_show_id"].ne("")
    )
    candidate_set.update(
        chunk.loc[valid, "tv_show_id"].tolist()
    )

candidates = sorted(candidate_set, key=int)
assert len(candidates) >= 5

pd.DataFrame(
    {"tv_show_id": candidates}
).to_csv(
    WORK / "final_candidate_ids.csv", index=False
)

template = pd.read_csv(
    BASE / "submission.csv",
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

assert list(template.columns) == ["user_id", "tv_show_id"]
assert template["user_id"].is_unique
assert template["user_id"].ne("").all()

test_users = template["user_id"].tolist()

# ------------------------------------------------------------------
# 4. Rebuild the same PPR graph on final training interactions.
# ------------------------------------------------------------------

users = sorted(
    final_interactions["user_id"].unique(), key=int
)
items = sorted(
    final_interactions["tv_show_id"].unique(), key=int
)

user_index = {
    user: index for index, user in enumerate(users)
}
item_index = {
    item: index for index, item in enumerate(items)
}

rows = final_interactions["user_id"].map(user_index).to_numpy()
columns = final_interactions["tv_show_id"].map(item_index).to_numpy()

binary = csr_matrix(
    (
        np.ones(len(final_interactions), dtype=np.float64),
        (rows, columns),
    ),
    shape=(len(users), len(items)),
)
assert binary.max() == 1.0

cooccurrence = (binary.T @ binary).tocsr()
cooccurrence.setdiag(0)
cooccurrence.eliminate_zeros()

item_user_counts = np.asarray(
    binary.sum(axis=0)
).ravel()
inverse_sqrt = 1.0 / np.sqrt(item_user_counts)

similarity = (
    diags(inverse_sqrt)
    @ cooccurrence
    @ diags(inverse_sqrt)
).tocsr()

del cooccurrence

graph_rows = []
graph_columns = []
graph_values = []

for row in range(len(items)):
    left = similarity.indptr[row]
    right = similarity.indptr[row + 1]

    neighbors = similarity.indices[left:right]
    weights = similarity.data[left:right]

    order = np.lexsort(
        (neighbors, -weights)
    )[:TOP_NEIGHBORS]

    graph_rows.extend([row] * len(order))
    graph_columns.extend(neighbors[order].tolist())
    graph_values.extend(weights[order].tolist())

graph = csr_matrix(
    (graph_values, (graph_rows, graph_columns)),
    shape=(len(items), len(items)),
)
graph = graph.maximum(graph.T).tocsr()
graph.setdiag(0)
graph.eliminate_zeros()

del similarity, graph_rows, graph_columns, graph_values

degrees = np.asarray(graph.sum(axis=1)).ravel()
dangling = degrees == 0
inverse_degree = np.zeros_like(degrees)
inverse_degree[~dangling] = 1.0 / degrees[~dangling]

transition = (diags(inverse_degree) @ graph).tocsr()

row_sums = np.asarray(
    transition.sum(axis=1)
).ravel()
assert np.allclose(row_sums[~dangling], 1.0)
assert np.allclose(row_sums[dangling], 0.0)

history = csr_matrix(
    (
        np.log1p(
            final_interactions["watch_count"].to_numpy(
                dtype=np.float64
            )
        ),
        (rows, columns),
    ),
    shape=binary.shape,
)

popularity = (
    final_interactions.groupby("tv_show_id")["watch_count"]
    .sum()
    .to_dict()
)
popularity_order = sorted(
    candidates,
    key=lambda item: (
        -int(popularity.get(item, 0)), int(item)
    ),
)

eligible_nodes = np.array(
    [
        index for index, item in enumerate(items)
        if item in candidate_set
    ],
    dtype=np.int64,
)

warm_users = [
    user for user in test_users if user in user_index
]
cold_users = [
    user for user in test_users if user not in user_index
]

print(
    f"\nFinal graph: {len(items):,} nodes, "
    f"{graph.nnz // 2:,} undirected edges",
    flush=True,
)
print(
    f"Test candidates: {len(candidates):,} | "
    f"Warm users: {len(warm_users)} | "
    f"Cold users: {len(cold_users)}",
    flush=True,
)

# ------------------------------------------------------------------
# 5. Generate final recommendations.
# ------------------------------------------------------------------

predictions = {}
convergence_rows = []
recommendation_rows = []

for offset in range(0, len(warm_users), BATCH_SIZE):
    batch = warm_users[offset:offset + BATCH_SIZE]

    restart = history[
        [user_index[user] for user in batch]
    ].T.toarray()
    restart /= restart.sum(axis=0, keepdims=True)

    p = restart.copy()

    for iteration in range(1, MAX_ITERATIONS + 1):
        dangling_mass = p[dangling].sum(axis=0)

        next_p = (
            FINAL_ALPHA * (transition.T @ p)
            + restart * (
                1.0 - FINAL_ALPHA
                + FINAL_ALPHA * dangling_mass
            )[None, :]
        )

        residuals = np.abs(next_p - p).sum(axis=0)
        p = next_p

        if float(residuals.max()) < TOLERANCE:
            break

    assert np.isfinite(p).all()
    assert p.min() >= -1e-12
    assert np.allclose(p.sum(axis=0), 1.0, atol=1e-8)

    if not (residuals < TOLERANCE).all():
        raise RuntimeError(
            "Final PPR did not converge for every user in a batch."
        )

    for column, user in enumerate(batch):
        scores = p[:, column]
        positive = eligible_nodes[
            scores[eligible_nodes] > 0
        ]

        order = np.lexsort(
            (positive, -scores[positive])
        )
        selected_nodes = positive[order[:5]]
        selected = [items[index] for index in selected_nodes]
        personalized_slots = len(selected)

        score_lookup = {
            items[index]: float(scores[index])
            for index in selected_nodes
        }

        for item in popularity_order:
            if len(selected) == 5:
                break
            if item not in selected:
                selected.append(item)

        predictions[user] = selected

        convergence_rows.append(
            {
                "user_id": user,
                "iterations": iteration,
                "residual": float(residuals[column]),
                "converged": True,
                "fallback_slots": 5 - personalized_slots,
            }
        )

        for rank, item in enumerate(selected, start=1):
            recommendation_rows.append(
                {
                    "user_id": user,
                    "rank": rank,
                    "tv_show_id": item,
                    "ppr_score": score_lookup.get(item, 0.0),
                    "source": (
                        "ppr"
                        if rank <= personalized_slots
                        else "popularity_fallback"
                    ),
                }
            )

    print(
        f"Final warm users: "
        f"{min(offset + BATCH_SIZE, len(warm_users))}"
        f"/{len(warm_users)}",
        flush=True,
    )

for user in cold_users:
    predictions[user] = popularity_order[:5]

    for rank, item in enumerate(predictions[user], start=1):
        recommendation_rows.append(
            {
                "user_id": user,
                "rank": rank,
                "tv_show_id": item,
                "ppr_score": 0.0,
                "source": "popularity_fallback",
            }
        )

# ------------------------------------------------------------------
# 6. Validate and save the exact submission file.
# ------------------------------------------------------------------

assert set(predictions) == set(test_users)

for user in test_users:
    ids = predictions[user]
    assert len(ids) == 5
    assert len(set(ids)) == 5
    assert all(
        item.isdigit()
        and int(item) > 0
        and item in candidate_set
        for item in ids
    )

submission = pd.DataFrame(
    {
        "user_id": test_users,
        "tv_show_id": [
            " ".join(predictions[user])
            for user in test_users
        ],
    }
)

submission_path = WORK / "submission_ppr_final.csv"
submission.to_csv(submission_path, index=False)

read_back = pd.read_csv(
    submission_path,
    dtype=str,
    keep_default_na=False,
)
assert list(read_back.columns) == ["user_id", "tv_show_id"]
assert read_back["user_id"].tolist() == test_users
assert read_back.equals(submission)

file_hash = hashlib.sha256(
    submission_path.read_bytes()
).hexdigest()

pd.DataFrame(convergence_rows).to_csv(
    WORK / "final_ppr_convergence.csv", index=False
)
pd.DataFrame(recommendation_rows).to_csv(
    WORK / "final_recommendations_detail.csv", index=False
)

joblib.dump(
    {
        "item_ids": items,
        "transition": transition,
        "dangling": dangling,
        "alpha": FINAL_ALPHA,
        "top_neighbors": TOP_NEIGHBORS,
    },
    WORK / "final_ppr_model.joblib",
)

summary = {
    "method": "PPR with train-popularity fallback",
    "alpha": FINAL_ALPHA,
    "alpha_source": "Fixed from local validation; no retuning",
    "training_window": {
        "start": "2020-03-09T00:00:00",
        "end_exclusive": "2020-07-27T00:00:00",
    },
    "schedule_processing": schedule_stats,
    "viewing_processing": viewing_stats,
    "interactions": final_interaction_stats,
    "graph_nodes": len(items),
    "graph_undirected_edges": int(graph.nnz // 2),
    "dangling_nodes": int(dangling.sum()),
    "candidate_programmes": len(candidates),
    "test_users": len(test_users),
    "warm_users": len(warm_users),
    "cold_users": len(cold_users),
    "nonconverged_warm_users": 0,
    "submission_file": submission_path.name,
    "submission_sha256": file_hash,
    "submission_checks_passed": True,
    "kaggle_submission_sent": False,
    "kaggle_public_score": None,
    "runtime_seconds": round(time.perf_counter() - started, 3),
}

with (WORK / "final_submission_summary.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(
        summary,
        handle,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("\nFINAL SUBMISSION FILE READY")
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\nSubmission preview:")
print(submission.head().to_string(index=False))

print("\nFile to upload to Kaggle:")
print(submission_path.name)
print("\nNo submission has been sent automatically.")

FINAL TRAINING AND SUBMISSION BUILD
Fixed alpha: 0.7
Loaded: export_arh_11-20-final.csv
Loaded: export_arh_21-30-final.csv

Final train schedule:
{
  "sources": [
    {
      "file": "export_arh_11-20-final.csv",
      "records": 1074235,
      "excluded_nonpositive_duration": 1,
      "excluded_zero_id_after_duration_filter": 958621
    },
    {
      "file": "export_arh_21-30-final.csv",
      "records": 1074394,
      "excluded_nonpositive_duration": 0,
      "excluded_zero_id_after_duration_filter": 952889
    }
  ],
  "duplicates_removed": 0,
  "boundary_crossing_excluded": 65,
  "entirely_outside": 281,
  "retained_airings": 236772
}
Views read: 1,000,000
Views read: 2,000,000
Views read: 3,000,000
Views read: 4,000,000
Views read: 5,000,000
{
  "records": 5821987,
  "duration_mismatch_excluded": 117,
  "irrelevant_channel_excluded_after_mismatch": 1103918,
  "affected_users": 113,
  "affected_user_examples": [
    "10059425067810734795",
    "10062914759109347465",
    "10162430

In [19]:
import os
from pathlib import Path

from IPython.display import FileLink, display

os.chdir("/kaggle/working")

submission_file = Path("submission_ppr_final.csv")

if not submission_file.is_file():
    raise FileNotFoundError("submission_ppr_final.csv was not found.")

print(f"Ready to download: {submission_file.name}")
print(f"File size: {submission_file.stat().st_size:,} bytes")

display(
    FileLink(
        "submission_ppr_final.csv",
        result_html_prefix="Download final submission: ",
    )
)

Ready to download: submission_ppr_final.csv
File size: 75,716 bytes


/kaggle/working/submission_ppr_final.csv

In [21]:
from pathlib import Path

path = Path("/kaggle/working/ppr_validation_summary.json")

if path.is_file():
    print(path.read_text(encoding="utf-8"))
else:
    print("Файл не найден. Нужен вывод ячейки с результатами validation.")

{
  "method": "ppr",
  "selected_alpha": 0.7,
  "selection": "Validation MAP@5; smaller alpha for exact ties",
  "cohort_size": 1238,
  "warm_users": 1224,
  "cold_users": 14,
  "map_at_5": 0.3764308472446598,
  "warm_map_at_5": 0.3788573438634713,
  "cold_map_at_5": 0.1642857142857143,
  "graph": {
    "nodes": 2489,
    "undirected_edges": 81292,
    "dangling_nodes": 0,
    "top_neighbors_before_symmetric_union": 50,
    "edge_weight": "Shared distinct train users divided by sqrt(item user count product)",
    "node_universe": "Programmes watched in train",
    "validation_candidates_absent_from_graph": 4498
  },
  "restart": "Normalized log1p(train watch_count)",
  "dangling_policy": "Redistribute mass to user restart vector",
  "tolerance": 1e-08,
  "max_iterations": 500,
  "batch_size": 64,
  "iteration_reporting": "Users in a batch share its stopping iteration; residuals are measured individually",
  "repeats_allowed": true,
  "fallback": "Shared train-popularity fallback",
  "s

In [22]:
from pathlib import Path

work = Path("/kaggle/working")

for path in sorted(work.iterdir()):
    if path.is_file() and path.suffix.lower() in {".json", ".csv"}:
        print(f"{path.name} — {path.stat().st_size:,} bytes")

boundary_and_anomaly_checks.json — 2,121 bytes
content_validation_per_user.csv — 104,986 bytes
content_validation_recommendations.csv — 535,546 bytes
content_validation_summary.json — 1,278 bytes
data_diagnostic_summary.json — 6,163 bytes
duration_mismatches.csv — 10,555 bytes
example_users.csv — 266 bytes
example_users_recommendations.csv — 4,210 bytes
example_users_train_history.csv — 9,724 bytes
example_users_validation_actual.csv — 13,932 bytes
final_candidate_ids.csv — 59,618 bytes
final_ppr_convergence.csv — 66,258 bytes
final_recommendations_detail.csv — 346,348 bytes
final_submission_summary.json — 2,309 bytes
final_train_interactions.csv — 7,088,860 bytes
final_train_item_metadata.csv — 1,752,666 bytes
interaction_build_summary.json — 3,418 bytes
july_boundary_conflicts.csv — 214 bytes
july_boundary_crossings.csv — 62,538 bytes
july_boundary_records.csv — 6,572,203 bytes
july_cross_file_duplicate_airings.csv — 60 bytes
may_boundary_conflicts.csv — 214 bytes
may_boundary_crossi

In [24]:

from pathlib import Path

work = Path("/kaggle/working")

files = [
    "validation_method_comparison.csv",
    "popularity_validation_summary.json",
    "content_validation_summary.json",
    "interaction_build_summary.json",
    "boundary_and_anomaly_checks.json",
]

for filename in files:
    path = work / filename
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if path.is_file():
        print(path.read_text(encoding="utf-8"))
    else:
        print("Файл не найден.")


validation_method_comparison.csv
method,cohort_size,map_at_5,warm_map_at_5,cold_map_at_5,stage_runtime_seconds,selected_alpha
popularity,1238,0.2961265930712619,0.29763457697893975,0.1642857142857143,0.455,
content_based,1238,0.1623815293484114,0.16235974945533768,0.1642857142857143,5.663,
ppr,1238,0.3764308472446598,0.3788573438634713,0.1642857142857143,19.667,0.7


popularity_validation_summary.json
{
  "method": "popularity",
  "popularity_definition": "Sum of qualifying train-airing views per programme",
  "metric": "MAP@5",
  "ap_denominator": "min(5, number of unique target programmes)",
  "target_definition": "Top five by validation watch_count; frequency ties use ascending numeric programme ID",
  "cohort_definition": "Submission-template users with nonempty validation targets, including users without train interactions",
  "repeats_of_previously_watched_items_allowed": true,
  "submission_users": 1260,
  "cohort_size": 1238,
  "submission_users_without_validation_target": 22,

In [26]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
import os

work = Path("/kaggle/working")

patterns = [
    "*summary.json",
    "boundary_and_anomaly_checks.json",
    "validation_method_comparison.csv",
    "ppr_alpha_sweep.csv",
    "*per_user.csv",
    "example_users*.csv",
    "final_ppr_convergence.csv",
    "submission_ppr_final.csv",
]

files = sorted({
    path
    for pattern in patterns
    for path in work.glob(pattern)
    if path.is_file()
})

archive = work / "experiment_evidence.zip"

with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in files:
        bundle.write(path, arcname=path.name)

with ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert "submission_ppr_final.csv" in bundle.namelist()

print(f"Files included: {len(files)}")
print(f"Archive size: {archive.stat().st_size:,} bytes")

os.chdir(work)
display(FileLink(archive.name))

Files included: 22
Archive size: 316,310 bytes


/kaggle/working/experiment_evidence.zip

In [27]:
from pathlib import Path

work = Path("/kaggle/working")

filenames = [
    "example_users.csv",
    "example_users_recommendations.csv",
    "example_users_train_history.csv",
    "example_users_validation_actual.csv",
]

for filename in filenames:
    path = work / filename

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if path.is_file():
        print(path.read_text(encoding="utf-8"))
    else:
        print("Файл не найден.")


example_users.csv
viewer_type,user_id,train_airing_views,train_unique_programmes,validation_unique_programmes,validation_target_ids
heavy,10500740270951698446,904,97,123,12001682 12002856 12001697 2400480 12002355
light,12453026253397466416,1,1,16,240081 2400480 5200303 20088 200337


example_users_recommendations.csv
viewer_type,user_id,method,rank,tv_show_id,title,category,in_train_history,in_validation_top5_target,validation_watch_count,train_popularity,user_ap_at_5
heavy,10500740270951698446,popularity,1,2400480,ТСН. Неделя.,Инфо,True,True,36,39939,0.3333333333333333
heavy,10500740270951698446,popularity,2,240081,Факты. День.,Инфо,True,False,14,27032,0.3333333333333333
heavy,10500740270951698446,popularity,3,12001682,М/с 'Губка Боб Квадратные Штаны'.,Детям,True,True,198,26195,0.3333333333333333
heavy,10500740270951698446,popularity,4,20088,'Завтрак с 1+1'. Информационно-развлекательная программа.,Развлечения,True,False,24,12796,0.3333333333333333
heavy,10500740270951698446,popular